In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# ---------------------------------------------------
# CONFIG (use raw strings for Windows paths)
# ---------------------------------------------------
NDVI_DIR = Path(r"C:\Drought\Regridding and data clipping\MODIS_NDVI")
NDVI_DERIVED = NDVI_DIR / "NDVI_VCI_monthly_India_0p05deg_2001present.nc"
NDVI_RAW     = NDVI_DIR / "MODIS_NDVI_India_0p05deg_2001-2023.nc"

LST_FILE = Path(r"C:\Drought\Regridding and data clipping\MODIS_LST") / "MODIS_LST_mean_India_0p05deg_2001-2023.nc"

OUT_FILE = NDVI_DIR / "TVDI_monthly_India_0p05deg_2001present.nc"

BASE_START = "2001-01"
BASE_END   = "2020-12"

# NDVI–LST triangle settings
NDVI_RANGE = (0.1, 0.9)   # ignore very low/high NDVI tails
N_BINS     = 20           # NDVI bins to estimate edges
WET_Q      = 0.05         # 5th percentile LST per NDVI bin
DRY_Q      = 0.95         # 95th percentile LST per NDVI bin
MIN_SAMPLES_PER_BIN = 200 # skip bins with too few points

# QC clips
NDVI_CLIP = (-0.2, 1.0)
LSTC_CLIP = (-30.0, 70.0)  # °C

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------
def to_month_end(da: xr.DataArray) -> xr.DataArray:
    """Make timestamps month-end, sorted, unique."""
    t = pd.to_datetime(da["time"].values)
    mend = pd.PeriodIndex(t, freq="M").to_timestamp("M")
    da2 = da.assign_coords(time=mend)
    # drop potential duplicates, sort
    _, idx = np.unique(da2["time"].values, return_index=True)
    return da2.isel(time=np.sort(idx)).sortby("time")

def pick_var(ds: xr.Dataset, prefs) -> xr.DataArray:
    """Pick the first variable in `prefs` that exists; else the only data_var; else raise."""
    for p in prefs:
        if p in ds:
            return ds[p]
    if len(ds.data_vars) == 1:
        return ds[list(ds.data_vars)[0]]
    raise ValueError(f"Could not find any of {prefs} in dataset; available: {list(ds.data_vars)}")

def to_celsius(lst_da: xr.DataArray) -> xr.DataArray:
    """Convert LST to °C if units suggest Kelvin or values are large."""
    u = str(getattr(lst_da, "units", "")).lower()
    if u.startswith("k") or float(lst_da.mean().values) > 200.0:
        return (lst_da - 273.15).assign_attrs(units="degC")
    return lst_da.assign_attrs(units="degC")

def fit_linear_edges_for_month(ndvi_m: xr.DataArray, lstc_m: xr.DataArray):
    """
    Fit wet & dry linear edges LST = a*NDVI + b for a given calendar month,
    using all baseline years for that month. Returns (aw,bw, ad,bd).
    """
    # mask finite & NDVI range
    mask = np.isfinite(ndvi_m) & np.isfinite(lstc_m)
    nd = ndvi_m.where(mask).clip(*NDVI_RANGE)
    ls = lstc_m.where(mask)

    # stack space & time
    ndv = nd.stack(points=("time", "lat", "lon")).dropna("points")
    lst = ls.stack(points=("time", "lat", "lon")).dropna("points")
    if ndv.size == 0:
        return np.nan, np.nan, np.nan, np.nan

    # NDVI bins
    bins = np.linspace(NDVI_RANGE[0], NDVI_RANGE[1], N_BINS + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    wet_vals, dry_vals, x_centers = [], [], []

    # quantiles per bin
    for i in range(N_BINS):
        sel = (ndv >= bins[i]) & (ndv < bins[i+1])
        n = sel.sum().item()
        if n < MIN_SAMPLES_PER_BIN:
            continue
        x_centers.append(centers[i])
        wet_vals.append(lst.where(sel).quantile(WET_Q, dim="points").item())
        dry_vals.append(lst.where(sel).quantile(DRY_Q, dim="points").item())

    if len(x_centers) < 2:
        # fallback: constant edges using overall quantiles
        w = float(lst.quantile(WET_Q, dim="points").values)
        d = float(lst.quantile(DRY_Q, dim="points").values)
        if np.isfinite(w) and np.isfinite(d) and (d - w) > 0.1:
            return 0.0, w, 0.0, d
        return np.nan, np.nan, np.nan, np.nan

    x = np.array(x_centers)
    w = np.array(wet_vals)
    d = np.array(dry_vals)

    # linear fits
    aw, bw = np.polyfit(x, w, 1)
    ad, bd = np.polyfit(x, d, 1)

    # sanity: ensure dry above wet on average
    if np.nanmean(ad*x + bd - (aw*x + bw)) <= 0:
        # widen slightly
        bd = bw + 0.5
        ad = aw
    return aw, bw, ad, bd

def build_edge_coeffs(ndvi: xr.DataArray, lstc: xr.DataArray, base_start: str, base_end: str):
    """For each calendar month, compute linear edge coefficients over baseline."""
    nd_base  = ndvi.sel(time=slice(base_start, base_end))
    lst_base = lstc.sel(time=slice(base_start, base_end))
    aw = np.full(12, np.nan); bw = np.full(12, np.nan)
    ad = np.full(12, np.nan); bd = np.full(12, np.nan)
    for m in range(1, 13):
        nd_m  = nd_base.where(nd_base["time.month"] == m, drop=True)
        ls_m  = lst_base.where(lst_base["time.month"] == m, drop=True)
        aw[m-1], bw[m-1], ad[m-1], bd[m-1] = fit_linear_edges_for_month(nd_m, ls_m)
        print(f"Month {m:02d}: wet LST = {aw[m-1]:+.3f}*NDVI {bw[m-1]:+.3f}, "
              f"dry LST = {ad[m-1]:+.3f}*NDVI {bd[m-1]:+.3f}")
    coords = {"month": np.arange(1, 13, dtype=int)}
    return (xr.DataArray(aw, coords=coords, dims=("month",), name="aw"),
            xr.DataArray(bw, coords=coords, dims=("month",), name="bw"),
            xr.DataArray(ad, coords=coords, dims=("month",), name="ad"),
            xr.DataArray(bd, coords=coords, dims=("month",), name="bd"))

def compute_tvdi(ndvi: xr.DataArray, lstc: xr.DataArray, aw, bw, ad, bd):
    """Apply TVDI for each time using coefficients of its calendar month."""
    month_idx = xr.DataArray(ndvi["time.month"].values, coords={"time": ndvi["time"]}, dims=("time",))
    aw_t = aw.sel(month=month_idx); bw_t = bw.sel(month=month_idx)
    ad_t = ad.sel(month=month_idx); bd_t = bd.sel(month=month_idx)
    lst_wet = aw_t * ndvi + bw_t
    lst_dry = ad_t * ndvi + bd_t
    denom   = lst_dry - lst_wet
    tvdi = ((lstc - lst_wet) / denom).clip(0.0, 1.0)
    return tvdi.rename("TVDI").assign_attrs(
        long_name="Temperature Vegetation Dryness Index (climatological edges per month)",
        units="1",
        definition="(LST - LST_wet(NDVI)) / (LST_dry(NDVI) - LST_wet(NDVI))"
    )

# ---------------------------------------------------
# Load NDVI
# ---------------------------------------------------
if NDVI_DERIVED.exists():
    ds_nd = xr.open_dataset(NDVI_DERIVED)
    ndvi  = ds_nd["NDVI"] if "NDVI" in ds_nd else pick_var(ds_nd, ["NDVI", "ndvi"])
else:
    ds_nd = xr.open_dataset(NDVI_RAW)
    ndvi  = pick_var(ds_nd, ["ndvi", "NDVI"])
ndvi = to_month_end(ndvi).clip(*NDVI_CLIP).assign_attrs(long_name="NDVI (QC-clipped)")

# Load LST
ds_lst = xr.open_dataset(LST_FILE)
lst = pick_var(ds_lst, ["LST", "lst", "LST_mean", "LST_Mean", "Ts", "LST_Day", "LST_Night"])
lst = to_month_end(lst)
lstC = to_celsius(lst).clip(*LSTC_CLIP).rename("LST_C").assign_attrs(long_name="LST (°C, QC-clipped)")

# Align time
common_time = np.intersect1d(ndvi["time"].values, lstC["time"].values)
ndvi = ndvi.sel(time=common_time)
lstC = lstC.sel(time=common_time)

# ---------------------------------------------------
# Build monthly edges from baseline & compute TVDI
# ---------------------------------------------------
print("Fitting NDVI–LST edges on 2001–2020 baseline …")
aw, bw, ad, bd = build_edge_coeffs(ndvi, lstC, BASE_START, BASE_END)

print("Computing TVDI …")
TVDI = compute_tvdi(ndvi, lstC, aw, bw, ad, bd)

# ---------------------------------------------------
# Save
# ---------------------------------------------------
print(f"Writing {OUT_FILE} …")
ds_out = xr.Dataset(
    data_vars=dict(
        TVDI=TVDI,
        NDVI=ndvi.rename("NDVI"),
        LST_C=lstC,
        aw=aw, bw=bw, ad=ad, bd=bd
    ),
    attrs=dict(
        title="Monthly TVDI from MODIS NDVI and LST (0.05°, India)",
        summary="Climatological NDVI–LST triangles (2001–2020) per calendar month; TVDI in [0,1].",
        conventions="CF-1.8",
        crs="EPSG:4326",
        spatial_resolution="0.05 degree",
        temporal_aggregation="monthly",
        history="Generated by make_tvdi_from_ndvi_lst.py",
        baseline=f"{BASE_START}..{BASE_END}",
        triangle_method=f"{N_BINS} NDVI bins, wet={int(WET_Q*100)}th pct, dry={int(DRY_Q*100)}th pct, min {MIN_SAMPLES_PER_BIN} samples per bin",
        ndvi_range=f"{NDVI_RANGE[0]}..{NDVI_RANGE[1]}",
    ),
)
comp = dict(zlib=True, complevel=4)
encoding = {v: comp for v in ds_out.data_vars}
encoding.update({"lat": {"dtype": "float32"}, "lon": {"dtype": "float32"}})
ds_out.to_netcdf(OUT_FILE, encoding=encoding)
print("Done.")


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Fitting NDVI–LST edges on 2001–2020 baseline …
Month 01: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 02: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 03: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 04: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 05: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 06: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 07: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 08: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 09: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 10: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 11: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Month 12: wet LST = +nan*NDVI +nan, dry LST = +nan*NDVI +nan
Computing TVDI …
Writing C:\Drought\Regridding and data clipping\MODIS_NDVI\TVDI_monthly_India_0p05deg_2001present.nc …
Done.
